In [48]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt
import mlflow
import subprocess

from pathlib import Path

from qrt.data import load_tr_data
from qrt.splitters import make_date_folds
from qrt.train import train_and_predict_cv, predict_positive_probability

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

In [49]:
X_tr, y_tr = load_tr_data(Path("../data/raw"))

In [50]:
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("qrt-aapf")

cv_params = {"n_splits": 8, "seed": 42}
folds = make_date_folds(X_tr, **cv_params)

X = X_tr.drop(columns=["TS", "GROUP", "ALLOCATION"])
y = (y_tr["target"] > 0).astype(int)

model = LGBMClassifier(
    n_estimators=200,
    num_leaves=7,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
)


In [51]:
root = Path("..").resolve()

with mlflow.start_run(run_name="lgbm_raw_benchmark") as run:
    # Record the experiment configuration
    mlflow.set_tags({
        "model": type(model).__name__,
        "reason": "First baseline using the 41 original numeric features",
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, text=True
        ).strip(),
    })

    mlflow.log_params(model.get_params())
    mlflow.log_params({
        "cv_n_splits": cv_params["n_splits"],
        "cv_seed": cv_params["seed"],
        "prediction_threshold": 0.5,
    })

    mlflow.log_dict({
        "features": X.columns.tolist(),
        "target": "target > 0",
        "training_files": ["X_train.csv", "y_train.csv"],
        "cv_method": "KFold on shuffled unique TS",
    }, "config.json")

    # Save the source files, including uncommitted changes
    for path in (root / "src/qrt").glob("*.py"):
        mlflow.log_artifact(str(path), artifact_path="source/qrt")

    for name in (
        "pyproject.toml",
        "uv.lock",
        "notebooks/010_benchmark.ipynb",
    ):
        mlflow.log_artifact(str(root / name), artifact_path="source")

    # Train and evaluate
    oof = train_and_predict_cv(model, X, y, folds, predict_positive_probability)
    assert oof.notna().all()

    y_pred = (oof >= 0.5).astype(int)
    scores = {"accuracy": accuracy_score(y, y_pred)}
    for fold in sorted(folds.unique()):
        rows = folds == fold
        scores[f"accuracy_fold_{fold}"] = accuracy_score(y[rows], y_pred[rows])

    mlflow.log_metrics(scores)

    # Save predictions and the exact fold assignments
    predictions = pd.DataFrame({
        "fold": folds,
        "target": y,
        "prediction": oof,
    })
    mlflow.log_text(predictions.to_csv(), "oof.csv")

pd.Series(scores)

🏃 View run lgbm_raw_benchmark at: http://127.0.0.1:5001/#/experiments/2/runs/c241c60532834b33949fe239bde94d4c
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


accuracy           0.522556
accuracy_fold_0    0.525241
accuracy_fold_1    0.524041
accuracy_fold_2    0.518249
accuracy_fold_3    0.524298
accuracy_fold_4    0.512719
accuracy_fold_5    0.525033
accuracy_fold_6    0.523069
accuracy_fold_7    0.528649
dtype: float64

In [53]:
group_scores = {}

for group in sorted(X_tr["GROUP"].unique()):
    rows = X_tr["GROUP"] == group
    group_scores[group] = accuracy_score(y[rows], y_pred[rows]) * 100

pd.Series(group_scores, name="accuracy")

1    52.176300
2    52.591753
3    51.990078
4    52.315906
Name: accuracy, dtype: float64